# Building and Deploying a Recommender with Text Features on MLServe.com

This notebook demonstrates how to train, deploy, and query a tabular recommendation model using the MLServe.com SDK.

We’ll simulate user–item interactions, engineer both structured (numeric, categorical) and unstructured (text) features, and train a LightGBM model wrapped in a scikit-learn pipeline.
The text field (e.g. product description) is vectorized with a lightweight TF-IDF encoder for simplicity and portability.

Once trained, the model is deployed to MLServe as a self-contained artifact, and can be queried via the SDK using a single call to `client.predict()`.

## Workflow overview

1. Generate synthetic data representing users, items, and interactions.
2. Preprocess data with a scikit-learn `ColumnTransformer` handling numeric, categorical, and text features.
3. Train a `LightGBMClassifier` to predict engagement (click / purchase).
4. Deploy the trained model to MLServe.com using the SDK (`client.deploy`).
5. Query the deployed endpoint with a real-time request simulating user + candidate items.

## Key characteristics

* Inference-first design — all preprocessing is contained in the model pipeline.
* Lightweight and fast — no deep learning dependencies or external embedding models.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier

from mlserve_sdk.client import MLServeClient
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
np.random.seed(42)
n_users, n_items, n_rows = 500, 300, 20000

user_df = pd.DataFrame({
    "user_id": np.arange(n_users),
    "age": np.random.randint(18, 60, n_users),
    "gender": np.random.choice(["M", "F"], n_users),
    "segment": np.random.choice(["A", "B", "C"], n_users)
})

item_df = pd.DataFrame({
    "item_id": np.arange(n_items),
    "category": np.random.choice(["books", "electronics", "fashion", "home"], n_items),
    "price": np.random.uniform(5, 200, n_items).round(2),
    "popularity": np.random.randint(1, 1000, n_items),
    "description": np.random.choice([
        "A thrilling adventure story.",
        "Comfortable and stylish everyday wear.",
        "Latest model smartphone with fast processor.",
        "Beautifully crafted piece for your home.",
        "Highly rated book by top author."
    ], n_items)
})

pairs = pd.DataFrame({
    "user_id": np.random.choice(user_df["user_id"], n_rows),
    "item_id": np.random.choice(item_df["item_id"], n_rows),
})

df = pairs.merge(user_df, on="user_id").merge(item_df, on="item_id")

segment_pref = {"A": "books", "B": "electronics", "C": "fashion"}
df["category_match"] = (df["segment"].map(segment_pref) == df["category"]).astype(int)
click_prob = 0.05 + 0.3 * df["category_match"] + 0.001 * (100 - df["price"])
df["clicked"] = (np.random.rand(len(df)) < click_prob.clip(0, 0.9)).astype(int)

df

,user_id,item_id,age,gender,segment,category,price,popularity,description,category_match,clicked
0,260,42,55,M,A,home,70.35,669,Highly rated book by top author.,0,1
1,260,258,55,M,A,electronics,75.05,905,Beautifully crafted piece for your home.,0,0
2,349,248,30,M,B,electronics,40.04,362,Highly rated book by top author.,1,1
3,351,73,53,M,A,home,126.08,147,Highly rated book by top author.,0,0
4,7,295,40,M,C,home,187.64,108,Latest model smartphone with fast processor.,0,0
...,...,...,...,...,...,...,...,...,...,...,...
19995,351,25,53,M,A,books,176.33,428,Comfortable and stylish everyday wear.,1,0
19996,426,272,56,M,A,electronics,38.10,154,Beautifully crafted piece for your home.,0,0
19997,426,158,56,M,A,electronics,32.79,221,Beautifully crafted piece for your home.,0,0
19998,205,188,39,F,C,fashion,71.19,804,A thrilling adventure story.,1,0


In [3]:
user_features = ["age", "gender", "segment"]
item_features = ["category", "price", "popularity", "description"]
all_features = user_features + item_features
label_col = "clicked"

X = df[all_features]
y = df[label_col]

categorical_features = ["gender", "segment", "category"]
numeric_features = ["age", "price", "popularity"]
text_features = ["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", StandardScaler(), numeric_features),
        ("text", TfidfVectorizer(max_features=200), "description"),
    ],
    remainder="drop"
)

model = LGBMClassifier(
    objective="binary",
    learning_rate=0.05,
    n_estimators=200,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

ranker = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])
ranker.fit(X, y)
print(f"✅ Model trained, train score: {ranker.score(X, y)}")

[LightGBM] [Info] Number of positive: 2695, number of negative: 17305
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000697 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 608
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134750 -> initscore=-1.859597
[LightGBM] [Info] Start training from score -1.859597
✅ Model trained, train score: 0.8836


/Users/nikosgavriil/Data Science/mls-project/notebooks/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
user_sample = user_df.sample(1).iloc[0].to_dict()
candidate_items = item_df.sample(5).to_dict(orient="records")

X_test = pd.DataFrame([{**user_sample, **item} for item in candidate_items])
scores = ranker.predict_proba(X_test)[:, 1]

print("\nTop predictions:")
for i, s in zip(candidate_items, scores):
    print(f"  Item {i['item_id']} ({i['category']}): {s:.3f} | {i['description']}")


Top predictions:
  Item 37 (fashion): 0.015 | Highly rated book by top author.
  Item 104 (home): 0.128 | Beautifully crafted piece for your home.
  Item 41 (home): 0.093 | Beautifully crafted piece for your home.
  Item 241 (books): 0.511 | Comfortable and stylish everyday wear.
  Item 18 (books): 0.369 | A thrilling adventure story.


/Users/nikosgavriil/Data Science/mls-project/notebooks/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
USERNAME = os.getenv("USERNAME")
TOKEN = os.getenv("TOKEN")

client = MLServeClient()
client.login(USERNAME, TOKEN)

In [6]:
try:
    lv=client.get_latest_version("recommender")
    next_version=lv["next_version"]
except:
    next_version="v1"

print(next_version)

v6


In [7]:
cols=list(X_test)
client.deploy(
    model=ranker,
    name="recommender",
    version="v6",
    features=cols,
    background_df=X_test,
    task_type='recommender',
    metrics={'score':ranker.score(X, y)}
)

/Users/nikosgavriil/Data Science/mls-project/notebooks/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'predict_url': 'https://mlserve.com/api/v1/predict/recommender/v6'}

In [8]:
%%time

user_sample = user_df.sample(1).iloc[0].to_dict()
candidate_items = item_df.sample(300).to_dict(orient="records")
X_test = pd.DataFrame([{**user_sample, **item} for item in candidate_items])

TEST_DATA = {
    "features": list(X_test),
    "inputs": [{'user':user_sample, 'items':candidate_items}]
}
preds = client.predict("recommender", "v6", TEST_DATA)
print("Predictions:", preds['predictions'][:10])

Predictions: [{'item_id': 190, 'score': 0.699}, {'item_id': 219, 'score': 0.578}, {'item_id': 18, 'score': 0.52}, {'item_id': 6, 'score': 0.499}, {'item_id': 57, 'score': 0.478}, {'item_id': 246, 'score': 0.474}, {'item_id': 285, 'score': 0.472}, {'item_id': 149, 'score': 0.461}, {'item_id': 171, 'score': 0.46}, {'item_id': 186, 'score': 0.449}]
CPU times: user 34.8 ms, sys: 4.13 ms, total: 38.9 ms
Wall time: 324 ms
